In [1]:
# =====================================================================
# CELLA IMPORTAZIONI LIBRERIE PER MAC
# =====================================================================
import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from keras.models import load_model

# Silenziamo i log inutili del Mac
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Blocca tutto tranne gli errori fatali
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# NOTA: Rimosso 'TF_CUDNN_USE_AUTOTUNE' perché sul tuo Mac non serve

import tensorflow as tf

# Altri silenziatori di log 
tf.get_logger().setLevel('ERROR')
tf.autograph.set_verbosity(0)

# =====================================================================
# MODIFICA 1: DISABILITARE LA GPU PER EVITARE I GRADIENTI NaN (ESPLOSIONE DELLA LOSS)
# =====================================================================
# Diciamo a TensorFlow di "nascondere" la GPU M1 (gestita da tensorflow-metal).
tf.config.set_visible_devices([], 'GPU')

# Riga di controllo per essere sicuri al 100% che abbia funzionato
print("Dispositivi di calcolo attivi:", tf.config.get_visible_devices())
# =====================================================================

# Import di Keras (lasciati identici a quelli del tuo collega)
from keras import layers, models, losses
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

Dispositivi di calcolo attivi: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [ ]:
# ==============================================================================
# DATA ENGINE V9 (Caricamento Globale in RAM)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Inizio caricamento ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
        mag_reshaped = mag.reshape(T, 1, 120, 18) 
        
        # Inizializza il background per l'EMA
        bg = np.copy(mag_reshaped[0])
        decluttered = np.zeros_like(mag_reshaped)
        
        # L'EMA viene calcolato qui, una volta per tutte, in perfetto ordine cronologico!
        for t in range(T):
            bg = alpha * mag_reshaped[t] + (1 - alpha) * bg
            decluttered[t] = np.abs(mag_reshaped[t] - bg)
        
        # Flatten delle coordinate e concatenazione con la mask (12 valori totali)
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pre-calcolati)")

    # Uniamo tutte le liste in due immensi tensori Numpy pronti per la GPU
    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# ==============================================================================
# 2. SPLIT STRATIFICATO RIGOROSO
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 3. ESECUZIONE DEL MOTORE (ATTENZIONE: Ci metterà ~1 minuto a caricare tutto in RAM)
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")

In [2]:
# ==============================================================================
# DATA ENGINE V9 (Canali I/Q separati + Correzione Geometria)
# ==============================================================================
def load_and_process_all_files(file_list, alpha=0.20):
    X_all, Y_all = [], []
    print(f"Inizio caricamento I/Q (36 canali) ed EMA Decluttering di {len(file_list)} file...")
    
    for i, file_path in enumerate(file_list):
        data = np.load(file_path)
        raw_iq = data['radar_cir_iq']   # Shape: (T, 6, 3, 120, 2)
        people_xy = data['people_xy']   
        people_mask = data['people_mask'] 
        T = raw_iq.shape[0]             
        
        # 1. EMA DECLUTTERING SUI CANALI COMPLESSI
        # Non calcoliamo più la magnitudo. Lavoriamo direttamente su I e Q.
        bg = np.copy(raw_iq[0])
        decluttered = np.zeros_like(raw_iq)
        
        for t in range(T):
            bg = alpha * raw_iq[t] + (1 - alpha) * bg
            # NIENTE np.abs()! Il segno di I e Q è vitale per la fase.
            decluttered[t] = raw_iq[t] - bg 
            
        # 2. TRASPOSIZIONE: Portiamo i 120 bin (asse 3) in posizione spaziale
        # Assi originali: 0=Time, 1=Radar, 2=Antenna, 3=Bin, 4=I/Q
        # Nuovi assi:     0=Time, 3=Bin, 1=Radar, 2=Antenna, 4=I/Q
        # Shape diventa: (T, 120, 6, 3, 2)
        decluttered_transposed = np.transpose(decluttered, (0, 3, 1, 2, 4))
        
        # 3. RESHAPE FINALE: Uniamo Radar(6) * Antenne(3) * canali_IQ(2) = 36 canali finali
        # Shape finale: (T, 1, 120, 36)
        decluttered_final = decluttered_transposed.reshape(T, 1, 120, 36)
        
        # Flatten delle coordinate
        flat_coords = people_xy.reshape(T, 8)
        combined_target = np.concatenate([flat_coords, people_mask], axis=1)

        X_all.append(decluttered_final)
        Y_all.append(combined_target)
        
        print(f"File {i+1}/{len(file_list)} processato. ({T} frame pronti)")

    X = np.concatenate(X_all, axis=0).astype(np.float32)
    Y = np.concatenate(Y_all, axis=0).astype(np.float32)
    return X, Y

# ==============================================================================
# 2. SPLIT STRATIFICATO RIGOROSO
# ==============================================================================
val_indices = [23, 20, 3, 15, 7, 11] 
train_indices = [22, 16, 17, 18, 19, 21, 0, 1, 2, 4, 12, 13, 14, 5, 6, 8, 9, 10]

#tutti_i_file = glob.glob("dataset/data/*.npz")
tutti_i_file = glob.glob("dataset/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

# ==============================================================================
# 3. ESECUZIONE DEL MOTORE (ATTENZIONE: Ci metterà ~1 minuto a caricare tutto in RAM)
# ==============================================================================
print("\n--- PREPARAZIONE TRAINING SET ---")
X_train, Y_train = load_and_process_all_files(train_files)

print("\n--- PREPARAZIONE VALIDATION SET ---")
X_val, Y_val = load_and_process_all_files(val_files)

print("\n==================================================")
print(f"DATI TOTALI PRONTI IN RAM!")
print(f"Totale FRAME individuali di Train:      {X_train.shape[0]}")
print(f"Totale FRAME individuali di Validation: {X_val.shape[0]}")
print("==================================================")


--- PREPARAZIONE TRAINING SET ---
Inizio caricamento I/Q (36 canali) ed EMA Decluttering di 18 file...
File 1/18 processato. (7500 frame pronti)
File 2/18 processato. (7500 frame pronti)
File 3/18 processato. (7500 frame pronti)
File 4/18 processato. (7500 frame pronti)
File 5/18 processato. (7500 frame pronti)
File 6/18 processato. (7500 frame pronti)
File 7/18 processato. (7500 frame pronti)
File 8/18 processato. (7500 frame pronti)
File 9/18 processato. (7500 frame pronti)
File 10/18 processato. (7500 frame pronti)
File 11/18 processato. (7500 frame pronti)
File 12/18 processato. (7500 frame pronti)
File 13/18 processato. (7500 frame pronti)
File 14/18 processato. (7500 frame pronti)
File 15/18 processato. (7500 frame pronti)
File 16/18 processato. (7500 frame pronti)
File 17/18 processato. (7500 frame pronti)
File 18/18 processato. (7500 frame pronti)

--- PREPARAZIONE VALIDATION SET ---
Inizio caricamento I/Q (36 canali) ed EMA Decluttering di 6 file...
File 1/6 processato. (7500

In [ ]:
# =====================================================================
# BULGARIAN SQUAT PER MAC
# =====================================================================
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    total_cost = coords_cost_norm + (1.5 * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [3]:
# =====================================================================
# BULGARIAN SQUAT PER MAC Modificato per I/Q superati 2 (forse anche 1)
# =====================================================================
import itertools
PERM_INDICES = tf.constant(list(itertools.permutations([0, 1, 2, 3])), dtype=tf.int32)

def hungarian_total_loss(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1) 
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)

    y_true_coords_exp = tf.expand_dims(y_true_coords, 1) 
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3]) 
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)

    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3]) 

    # =========================================================
    # IL SEGRETO E' QUI: Diamo un peso massiccio alle coordinate (es. 10.0)
    # per forzare la rete a localizzare con precisione.
    # =========================================================
    COORDS_WEIGHT = 10.0 
    MASK_WEIGHT = 1.0
    
    total_cost = (COORDS_WEIGHT * coords_cost_norm) + (MASK_WEIGHT * mask_cost_norm) 
    return tf.reduce_min(total_cost, axis=1) 

def hungarian_rmse_metres(y_true, y_pred):
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))

    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    
    min_coords_cost = tf.reduce_min(coords_cost, axis=1)
    
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    
    return tf.sqrt(min_coords_cost / num_valid_people)

def hungarian_mask_acc(y_true, y_pred):
    y_true_mask = tf.reshape(y_true[:, 8:], (-1, 4, 1))
    y_pred_mask = tf.reshape(y_pred[:, 8:], (-1, 4, 1))
    y_pred_mask_perm = tf.gather(y_pred_mask, PERM_INDICES, axis=1)
    
    y_true_coords = tf.reshape(y_true[:, :8], (-1, 4, 2))
    y_pred_coords = tf.reshape(y_pred[:, :8], (-1, 4, 2))
    y_pred_coords_perm = tf.gather(y_pred_coords, PERM_INDICES, axis=1)
    y_true_coords_exp = tf.expand_dims(y_true_coords, 1)
    y_true_mask_exp = tf.expand_dims(y_true_mask, 1)

    sq_diff = tf.square(y_true_coords_exp - y_pred_coords_perm)
    coords_cost = tf.reduce_sum(sq_diff * y_true_mask_exp, axis=[2, 3])
    num_valid_people = tf.reduce_sum(y_true_mask_exp[:, 0, :, :], axis=[1, 2]) + 1e-6
    coords_cost_norm = coords_cost / tf.expand_dims(num_valid_people, axis=-1)
    
    #bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm)
    y_pred_mask_perm_safe = tf.clip_by_value(y_pred_mask_perm, 1e-7, 1.0 - 1e-7)
    bce = tf.keras.backend.binary_crossentropy(y_true_mask_exp, y_pred_mask_perm_safe)
    mask_cost_norm = tf.reduce_mean(bce, axis=[2, 3])
    
    total_cost = coords_cost_norm + (1.5 * mask_cost_norm)
    best_perm_idx = tf.argmin(total_cost, axis=1, output_type=tf.int32)
    
    batch_size = tf.shape(y_pred)[0]
    gather_nd_indices = tf.stack([tf.range(batch_size, dtype=tf.int32), best_perm_idx], axis=1)
    best_mask_pred = tf.gather_nd(y_pred_mask_perm, gather_nd_indices)
    
    return tf.reduce_mean(tf.keras.metrics.binary_accuracy(y_true_mask, best_mask_pred))

In [ ]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V1-Heavy (Struttura Romana, Potenza Toscana) vecchia, nuova
# ==============================================================================
def build_eeai_model_v1_heavy(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- BLOCCO 1: Estrazione Base ---
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x)
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # Dimensione: 120 -> 60
    
    # --- BLOCCO 2: Livello Intermedio ---
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x)
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_4")(x) # Dimensione: 60 -> 30
    
    # --- BLOCCO 3: Feature di Alto Livello ---
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3a")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_5")(x)
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_6")(x)
    x = layers.Conv2D(128, (1, 3), padding='same', activation='relu', name="conv_3c")(x)
    
    # Compattazione salvavita per ESP32
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    
    # --- COLLO DI BOTTIGLIA PIÙ PROFONDO ---
    x = layers.Dense(256, activation='relu', name="features_deep2")(x)
    x = layers.Dropout(0.2, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    # --- OUTPUT MULTI-HEAD INVARIATI ---
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V1_Heavy")
    


# Inizializzazione del nuovo modello
model_heavy = build_eeai_model_v1_heavy()

# Compilazione 
model_heavy.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_heavy = ModelCheckpoint("eeai_best_model_romano_heavy.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO OTTIMIZZATO (CON SCHEDULER E EARLY STOP) ---")
history_light = model_heavy.fit(
    X_train, Y_train,                # Usiamo i tensori in RAM, non il generatore!
    validation_data=(X_val, Y_val),  # Usiamo i tensori in RAM!
    batch_size=32,                   # LA MAGIA AVVIENE QUI: 32 frame alla volta
    shuffle=True,                    # Rimescolamento perfetto per evitare bias
    epochs=EPOCHS,
    callbacks=[checkpoint_heavy, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [4]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V1-Heavy (36 Canali I/Q + Ottimizzazione Edge) 1
# ==============================================================================
def build_eeai_model_v1_heavy(n_radars=6, n_antennas=3, n_bins=120):
    # I canali ora sono 36 (6 * 3 * 2)
    input_channels = n_radars * n_antennas * 2 
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- BLOCCO 1: Estrazione Base ---
    # Kernel (1, 5) per scorrere SOLO sulla distanza, non sui canali
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1a")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) # 120 -> 60
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_1b")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # 60 -> 30
    
    # --- BLOCCO 2: Livello Intermedio (Separable) ---
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x) # 30 -> 15
    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_2")(x)
    
    # --- BLOCCO 3: Feature di Alto Livello (Separable per Edge) ---
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="conv_3a")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_5")(x) # 15 -> 7
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="conv_3b")(x)
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="conv_3c")(x)
    
    # Compattazione
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    
    # --- COLLO DI BOTTIGLIA ---
    x = layers.Dense(256, activation='relu', name="features_deep2")(x)
    x = layers.Dropout(0.2, name="drop_features")(x) 
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    # --- OUTPUT ---
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V1_Heavy_IQ")
    


# Inizializzazione del nuovo modello
model_heavy = build_eeai_model_v1_heavy()

# Compilazione 
model_heavy.compile(
    optimizer='adam',
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

checkpoint_heavy = ModelCheckpoint("eeai_best_model_romano_heavy.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO OTTIMIZZATO (CON SCHEDULER E EARLY STOP) ---")
history_light = model_heavy.fit(
    X_train, Y_train,                # Usiamo i tensori in RAM, non il generatore!
    validation_data=(X_val, Y_val),  # Usiamo i tensori in RAM!
    batch_size=32,                   # LA MAGIA AVVIENE QUI: 32 frame alla volta
    shuffle=True,                    # Rimescolamento perfetto per evitare bias
    epochs=EPOCHS,
    callbacks=[checkpoint_heavy, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")


--- INIZIO ADDESTRAMENTO OTTIMIZZATO (CON SCHEDULER E EARLY STOP) ---
Epoch 1/50
4215/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - hungarian_mask_acc: 0.6248 - hungarian_rmse_metres: 1.0252 - loss: 17.3771
Epoch 1: val_loss improved from None to 7.29506, saving model to eeai_best_model_romano_heavy.keras

Epoch 1: finished saving model to eeai_best_model_romano_heavy.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 46s 10ms/step - hungarian_mask_acc: 0.6695 - hungarian_rmse_metres: 0.8459 - loss: 10.5646 - val_hungarian_mask_acc: 0.7123 - val_hungarian_rmse_metres: 0.6879 - val_loss: 7.2951 - learning_rate: 0.0010
Epoch 2/50
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - hungarian_mask_acc: 0.7387 - hungarian_rmse_metres: 0.6550 - loss: 6.5131
Epoch 2: val_loss improved from 7.29506 to 5.27311, saving model to eeai_best_model_romano_heavy.keras

Epoch 2: finished saving model to eeai_best_model_romano_heavy.keras
4219/4219 ━━━━━━━━━━━━━━━━━━━━ 40s 10ms/step - hungarian_mask_acc: 0.7524 - hungarian_rmse_

In [ ]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V1-Heavy (36 Canali I/Q + Ottimizzazione Edge) 2
# ==============================================================================
def build_eeai_model_v2_local(n_radars=6, n_antennas=3, n_bins=120):
    # I canali sono 36 (6 * 3 * 2)
    input_channels = n_radars * n_antennas * 2 
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # --- BLOCCO 1: Estrazione Spaziale Primaria ---
    # Usiamo un kernel largo (1, 7) per catturare pattern più ampi sui bin
    x = layers.Conv2D(32, (1, 7), padding='same', activation='relu', name="conv_1")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) # 120 -> 60
    
    # --- BLOCCO 2: Raffinamento e Compattazione ---
    x = layers.Conv2D(64, (1, 5), padding='same', activation='relu', name="conv_2")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) # 60 -> 30
    
    # --- BLOCCO 3: Feature ad Alta Densità (Separable) ---
    x = layers.SeparableConv2D(128, (1, 3), padding='same', activation='relu', name="sep_conv_3")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_3")(x) # 30 -> 15
    
    # --- FOCUS SPAZIALE (Niente GlobalAveragePooling) ---
    # Preserviamo l'informazione del bin specifico appiattendo il tensore
    x = layers.Flatten(name="flatten_focus")(x)
    
    # --- COLLO DI BOTTIGLIA DENSO ---
    x = layers.Dense(256, activation='relu', name="dense_1")(x)
    x = layers.Dropout(0.3, name="drop_1")(x)
    common_feat = layers.Dense(128, activation='relu', name="dense_2")(x)

    # --- OUTPUT ---
    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)
    combined_output = layers.Concatenate(axis=1, name="combined_output")([coords_output, mask_output])

    return models.Model(inputs=inputs, outputs=combined_output, name="EEAI_Net_V2_Local")


# Inizializzazione del nuovo modello V2
model_v2 = build_eeai_model_v2_local()

# Compilazione 
model_v2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), # Partiamo con un LR standard
    loss=hungarian_total_loss, 
    metrics=[hungarian_rmse_metres, hungarian_mask_acc]
)

# Callbacks aggiornati per il nuovo nome
checkpoint_v2 = ModelCheckpoint("eeai_best_model_v2.keras", monitor="val_loss", save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1)

EPOCHS = 50 

print("\n--- INIZIO ADDESTRAMENTO V2 LOCAL FOCUS (I/Q Separati) ---")
history_v2 = model_v2.fit(
    X_train, Y_train,                
    validation_data=(X_val, Y_val),  
    batch_size=32,                   
    shuffle=True,                    
    epochs=EPOCHS,
    callbacks=[checkpoint_v2, reduce_lr, early_stop], 
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")

In [ ]:
# ==============================================================================
# MODEL SUMMARY PER EMBEDDED
# ==============================================================================

def embedded_summary(model, input_shape=(1, 120, 18)):
    
    # 2. Calcola i parametri statici (Flash)
    total_params = model.count_params()
    estimated_flash_kb = (total_params * 4) / 1024
    
   # 3. Calcola il picco di memoria dinamica (SRAM/Tensor Arena)
    max_layer_ram_kb = 0
    for layer in model.layers:
        # AGGIUNTO: Salta l'InputLayer o i layer senza output_shape per evitare l'AttributeError
        if layer.__class__.__name__ == 'InputLayer' or not hasattr(layer, 'output_shape'):
            continue
            
        output_shape = layer.output_shape
        if isinstance(output_shape, list):
            num_elements = sum([np.prod([dim for dim in shape[1:] if dim is not None]) for shape in output_shape])
        else:
            num_elements = np.prod([dim for dim in output_shape[1:] if dim is not None])
            
        layer_ram_kb = (num_elements * 4) / 1024
        if layer_ram_kb > max_layer_ram_kb:
            max_layer_ram_kb = layer_ram_kb

    input_elements = np.prod(input_shape)
    input_ram_kb = (input_elements * 4) / 1024
    peak_arena_kb = input_ram_kb + max_layer_ram_kb

    # 4. Stampa il verdetto 
    print("============================================")
    print("   REPORT REQUISITI HARDWARE (STIMA FLOAT32)   ")
    print("============================================")
    print(f" Memoria FLASH stimata : {estimated_flash_kb:.2f} KB  (Limite : < 800 KB)")
    print(f" Memoria SRAM stimata  : ~{peak_arena_kb:.2f} KB (Limite : < 400 KB)")
    print(" Operazioni Ricorrenti : ASSENTI (RNN/LSTM/GRU non rilevate)")
    print(" Nota sulla Quantizz.  : Raccomandata INT8 per ESP32-S3 (ridurrà la RAM di ~4x)")
    print("============================================\n")

embedded_summary(model_heavy)

In [6]:
# ==============================================================================
# VISUALIZZATORE 3.1 (Fix Output 12) nuovo
# ==============================================================================

#file_target = "dataset/data/window_000011.npz"
file_target = "dataset/window_000014.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V8)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    print("Caricamento dei pesi migliori dal file .keras ...")
    
    # Caricamento del modello 
    model_heavy = load_model(
        "eeai_best_model_romano_heavy.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_heavy.predict(decluttered, verbose=0)
    
    # ==========================================
    # IL FIX E' QUI: Slicing corretto per la V8
    # ==========================================
    # preds ha dimensione [T, 12]. 
    # Prendiamo le prime 8 colonne per le coordinate, e le ultime 4 per le maschere.
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # Titolo aggiornato
            ax.set_title(f"Radar V9 | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (V8)...
Caricamento dei pesi migliori dal file .keras ...


ValueError: Input 0 with name 'radar_input' of layer 'EEAI_Net_V1_Heavy_IQ' is incompatible with the layer: expected shape=(None, 1, 120, 36), found shape=(32, 1, 120, 18)

In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.1 (Fix Output 12) nuovo, I/Q separati 1
# ==============================================================================

#file_target = "dataset/data/window_000011.npz"
file_target = "dataset/window_000014.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri I/Q in corso (V9)...")
    # EMA sui canali I/Q separati
    decluttered = np.zeros_like(raw_iq)
    bg = np.copy(raw_iq[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * raw_iq[t] + (1 - alpha) * bg
        decluttered[t] = raw_iq[t] - bg
        
    # Transpose e Reshape finale a 36 canali
    decluttered = np.transpose(decluttered, (0, 3, 1, 2, 4)).reshape(T, 1, 120, 36)

    print("Caricamento dei pesi migliori dal file .keras ...")
    
    # Caricamento del modello 
    model_heavy = load_model(
        "eeai_best_model_romano_heavy.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_heavy.predict(decluttered, verbose=0)
    
    # ==========================================
    # IL FIX E' QUI: Slicing corretto per la V8
    # ==========================================
    # preds ha dimensione [T, 12]. 
    # Prendiamo le prime 8 colonne per le coordinate, e le ultime 4 per le maschere.
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # Titolo aggiornato
            ax.set_title(f"Radar V9 | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri I/Q in corso (V9)...
Caricamento dei pesi migliori dal file .keras ...
Dati pronti! Inizializzazione Radar...


In [8]:
# ==============================================================================
# VISUALIZZATORE 3.1 (Fix Output 12) nuovo, I/Q separati 2
# ==============================================================================

#file_target = "dataset/data/window_000011.npz"
file_target = "dataset/window_000014.npz"
if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri I/Q in corso (V9)...")
    # EMA sui canali I/Q separati
    decluttered = np.zeros_like(raw_iq)
    bg = np.copy(raw_iq[0])
    alpha = 0.20
    for t in range(T):
        bg = alpha * raw_iq[t] + (1 - alpha) * bg
        decluttered[t] = raw_iq[t] - bg
        
    # Transpose e Reshape finale a 36 canali
    decluttered = np.transpose(decluttered, (0, 3, 1, 2, 4)).reshape(T, 1, 120, 36)

    print("Caricamento dei pesi migliori dal file .keras ...")
    
    # Caricamento del nuovo modello V2
    model_heavy = load_model(
        "eeai_best_model_v2.keras",
        custom_objects={
            "hungarian_total_loss": hungarian_total_loss,
            "hungarian_rmse_metres": hungarian_rmse_metres, 
            "hungarian_mask_acc": hungarian_mask_acc
        }
    )

    preds = model_heavy.predict(decluttered, verbose=0)
    
    # ==========================================
    # IL FIX E' QUI: Slicing corretto per la V8
    # ==========================================
    # preds ha dimensione [T, 12]. 
    # Prendiamo le prime 8 colonne per le coordinate, e le ultime 4 per le maschere.
    p_coords = preds[:, :8].reshape(T, 4, 2)
    p_mask = preds[:, 8:]
    
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            
            # Titolo aggiornato
            ax.set_title(f"Radar V9 | Frame: {frame_idx}/{T-1} | Window: 11", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri I/Q in corso (V9)...
Caricamento dei pesi migliori dal file .keras ...
Dati pronti! Inizializzazione Radar...
